# Actividad 5: Pipeline de scikit-learn con MLflow

Este notebook adapta el pipeline de predicción de demanda y separa el trabajo en etapas.
Usa **Experiments**, **Tracking**, **Models** y, de forma opcional, **Model Registry**.
Los datos son sintéticos; no se necesita un Feature Store para esta actividad.

Ejecuta las celdas en orden con el entorno de esta actividad. Desde esta carpeta:
```bash
uv sync --extra notebook
uv run --extra notebook jupyter lab pipeline_mlflow.ipynb
```

`src/data.py` contiene la generación de datos y `src/pipeline.py` conserva los
transformadores y funciones reutilizables. Aquí se muestra explícitamente la
integración con MLflow, sin ocultarla en una sola llamada al pipeline.


## 1. Importaciones y ubicación del proyecto

Se admite abrir Jupyter desde la carpeta de la actividad, desde `src/` o desde
la raíz del repositorio. No se instalan paquetes desde las celdas.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from IPython.display import display
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [p / "workshop/activity-05-databricks-mlflow" for p in list(candidates)]
PROJECT_DIR = next(
    (p for p in candidates if (p / "src/pipeline.py").is_file()), None
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Abre Jupyter dentro del repositorio de esta actividad.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data import generate_demand_dataset
from src.pipeline import (
    build_pipeline,
    compute_metrics,
    create_evaluation_figure,
    load_env_file,
    split_temporal_data,
)

print(f"Proyecto: {PROJECT_DIR.name} | MLflow: {mlflow.__version__}")


## 2. Configurar Experiments

Por defecto se usa SQLite local. Aunque haya credenciales de Databricks en `.env`,
solo se envían datos a Databricks al cambiar `USE_DATABRICKS = True`.
Para ese modo, configura las credenciales fuera del notebook y define
`MLFLOW_EXPERIMENT_NAME` con una ruta de experimento de tu workspace
(por ejemplo, `/Users/tu_usuario/activity-05-demand-pipeline`).

`MLFLOW_TRACKING_URI` permite elegir otro backend local o un servidor MLflow.
Cada ejecución completa del notebook crea un run nuevo en el mismo experimento.


In [ ]:
USE_DATABRICKS = False
N_ROWS = 3000
SEED = 42
TEST_SIZE = 0.2
ALPHA = 1.0

load_env_file()
if USE_DATABRICKS:
    tracking_uri = "databricks"
    experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME")
    if not experiment_name or not experiment_name.startswith("/"):
        raise ValueError("Define MLFLOW_EXPERIMENT_NAME con una ruta de Databricks.")
else:
    tracking_uri = os.getenv("MLFLOW_TRACKING_URI", f"sqlite:///{PROJECT_DIR / 'mlflow.db'}")
    experiment_name = "activity-05-demand-notebook"

mlflow.set_tracking_uri(tracking_uri)
# El registro usa el mismo backend, salvo en Databricks (configuración explícita al final).
if not USE_DATABRICKS:
    mlflow.set_registry_uri(tracking_uri)
experiment = mlflow.set_experiment(experiment_name)
print(f"Experimento: {experiment.name} | ID: {experiment.experiment_id}")


## 3. Generar y explorar los datos

La variable objetivo es `demand`. `previous_days_demand` contiene la demanda del
día anterior y se obtiene con `shift(1)`. El generador elimina los días iniciales
sin historial. La semilla permite reproducir los datos.


In [ ]:
df = generate_demand_dataset(n_rows=N_ROWS, seed=SEED)
assert df["date"].is_monotonic_increasing
assert not df.isna().any().any()
display(df.head())
display(df.describe())


## 4. Separar entrenamiento y prueba

Se reserva el último 20 % para prueba, sin mezclar fechas. Las transformaciones
se ajustarán exclusivamente con entrenamiento. Esta evaluación simula predicciones
diarias con la demanda real del día anterior disponible; para pronosticar varios
días hacia adelante se necesitaría construir ese historial de manera recursiva.
El conjunto de prueba no se usa para elegir `alpha`.


In [ ]:
X_train, X_test, y_train, y_test = split_temporal_data(df, test_size=TEST_SIZE)
assert X_train["date"].max() < X_test["date"].min()
display(pd.DataFrame({
    "partición": ["entrenamiento", "prueba"],
    "filas": [len(X_train), len(X_test)],
    "desde": [X_train["date"].min(), X_test["date"].min()],
    "hasta": [X_train["date"].max(), X_test["date"].max()],
}))


## 5. Construir y entrenar el pipeline

`DemandFeatures` deriva mes, años transcurridos y ventaja de precio del competidor.
`ColumnTransformer` escala variables numéricas, codifica el mes y conserva las
binarias. `Ridge` estima la demanda. Guardaremos este pipeline completo para que
la inferencia reciba las mismas columnas originales.


In [ ]:
pipeline = build_pipeline(alpha=ALPHA)
pipeline.fit(X_train, y_train)
display(pipeline)


## 6. Evaluar el modelo

RMSE y MAE se expresan en unidades de demanda (menor es mejor); R² mide el ajuste
(mayor es mejor). El gráfico permite revisar errores y residuos.


In [ ]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)
metrics = {
    **compute_metrics(y_train, y_train_pred, prefix="train_"),
    **compute_metrics(y_test, y_test_pred, prefix="test_"),
}
display(pd.DataFrame([
    {"partición": partition, **{m: metrics[f"{prefix}{m}"] for m in ["rmse", "mae", "r2"]}}
    for partition, prefix in [("entrenamiento", "train_"), ("prueba", "test_")]
]))
figure = create_evaluation_figure(y_test, y_test_pred)
display(figure)
plt.close(figure)


## 7. Registrar el experimento y el modelo en MLflow

El run almacena parámetros, métricas, un gráfico y predicciones de prueba.
**MLflow Models** empaqueta el pipeline con firma de entrada/salida y ejemplos.
Se incluye `src/` para que el transformador personalizado pueda importarse al
recuperar el modelo desde otro proceso. Las dependencias de inferencia se fijan
con las versiones del entorno utilizado.


In [ ]:
from importlib.metadata import version
import inspect

sample_input = X_train.head(5)
signature = infer_signature(sample_input, pipeline.predict(sample_input))
# MLflow 2 usa artifact_path; MLflow 3 incorpora name.
model_location = (
    {"name": "demand_pipeline_model"}
    if "name" in inspect.signature(mlflow.sklearn.log_model).parameters
    else {"artifact_path": "demand_pipeline_model"}
)

with mlflow.start_run(run_name=f"notebook_ridge_alpha_{ALPHA}") as run:
    mlflow.set_tags({
        "project": "activity-05-databricks-mlflow",
        "source": "pipeline_mlflow.ipynb",
        "framework": "scikit-learn",
        "environment": "Databricks" if USE_DATABRICKS else "Local",
        "evaluation": "temporal_one_day_ahead",
    })
    mlflow.log_params({
        "model_type": "Ridge", "alpha": ALPHA, "seed": SEED,
        "n_rows": N_ROWS, "test_size": TEST_SIZE,
        "train_samples": len(X_train), "test_samples": len(X_test),
        "pipeline_steps": "DemandFeatures -> ColumnTransformer -> Ridge",
    })
    mlflow.log_metrics(metrics)
    mlflow.log_figure(figure, "evaluation/residuals.png")
    predictions = pd.DataFrame({
        "date": X_test["date"], "actual": y_test, "predicted": y_test_pred,
    })
    mlflow.log_text(predictions.to_csv(index=False), "evaluation/predictions.csv")
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        **model_location,
        signature=signature,
        input_example=sample_input,
        serialization_format="cloudpickle",
        code_paths=[str(PROJECT_DIR / "src")],
        pip_requirements=[
            f"{package}=={version(package)}"
            for package in ["scikit-learn", "pandas", "numpy", "matplotlib", "cloudpickle"]
        ],
    )
    run_id = run.info.run_id
    model_uri = model_info.model_uri

print(f"Run ID: {run_id}")
print(f"Modelo: {model_uri}")


## 8. Consultar los runs del experimento

La tabla permite revisar el historial de ejecuciones. Comparar resultados sobre
prueba es descriptivo: para ajustar hiperparámetros se debe reservar una partición
de validación temporal adicional.


In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=20,
)
display(runs[[
    "run_id", "start_time", "params.alpha",
    "metrics.test_rmse", "metrics.test_mae", "metrics.test_r2",
]])


## 9. Cargar el modelo y comprobar la inferencia

Se recupera el modelo mediante su URI de MLflow. La comprobación confirma que el
pipeline serializado produce las mismas predicciones, incluyendo preprocesamiento.


In [ ]:
loaded_pipeline = mlflow.sklearn.load_model(model_uri)
new_data = X_test.head(5).copy()
loaded_predictions = loaded_pipeline.predict(new_data)
np.testing.assert_allclose(loaded_predictions, pipeline.predict(new_data))
display(new_data.assign(predicted_demand=loaded_predictions))
print("Verificación correcta: las predicciones coinciden después de cargar el modelo.")


## 10. Model Registry (opcional)

Activa `REGISTER_MODEL` para crear una versión del modelo. En local se utiliza el
registro de SQLite. En Databricks configura `REGISTRY_URI` (por ejemplo,
`databricks-uc`) y un nombre autorizado `catalog.schema.model`; el catálogo y
esquema deben existir. Registrar una versión no despliega un endpoint.


In [ ]:
REGISTER_MODEL = False
REGISTERED_MODEL_NAME = "sklearn_demand_prediction_pipeline"
REGISTRY_URI = "databricks-uc" if USE_DATABRICKS else tracking_uri

if REGISTER_MODEL:
    if USE_DATABRICKS and REGISTRY_URI == "databricks-uc" and len(REGISTERED_MODEL_NAME.split(".")) != 3:
        raise ValueError("Usa un nombre catalog.schema.model para Unity Catalog.")
    mlflow.set_registry_uri(REGISTRY_URI)
    model_version = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
    client = MlflowClient()
    print(f"Modelo registrado: {model_version.name} | Versión: {model_version.version}")
else:
    print("El modelo está guardado en MLflow Models. Registro de versión opcional desactivado.")


## Revisar los resultados

Para el backend local predeterminado, ejecuta desde la carpeta de la actividad:
```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```
Abre http://127.0.0.1:5000 y selecciona `activity-05-demand-notebook`.
Si elegiste otro backend, inicia la interfaz con esa misma URI.
En Databricks, abre **Experiments** en tu workspace y busca el nombre configurado.

Documentación oficial: [MLflow y scikit-learn](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html).
